# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
from project_utils import project_path

# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {
    "65_YEARS_OF_WEATHER_DATA_BANGLADESH_1948_2013_CSV": project_path(
        "65 Years of Weather Data Bangladesh (1948 - 2013).csv"
    ),
    "DHAKA_WEATHER_CSV": project_path("dhaka_weather.csv"),
}

# Data Preprocessing



In [ ]:
# Check for GPU

pass  # Environment setup moved to README; use local inputs.

In [ ]:
# Mount Google Drive

pass  # Environment setup moved to README; use local inputs.
pass  # Environment setup moved to README; use local inputs.

In [ ]:
# Load the dataset

import pandas as pd

df = pd.read_csv(INPUT_PATHS["65_YEARS_OF_WEATHER_DATA_BANGLADESH_1948_2013_CSV"])
df.head()

In [ ]:
# Get statistics of the dataset

df.describe()

In [ ]:
# Get info on the type of data in each column

df.info()

In [ ]:
# Filter out unnecessary columns

df_filtered = df.drop(
    [
        "ALT",
        "Unnamed: 0",
        "X_COR",
        "Y_COR",
        "Station Number",
        "LATITUDE",
        "LONGITUDE",
        "Period",
        "Wind Speed",
        "Bright Sunshine",
    ],
    axis=1,
)
df_filtered.head()

In [ ]:
# Rename columns

df_filtered.rename(
    columns={
        "Cloud Coverage": "Cloud Cover (Okta)",
        "Relative Humidity": "Relative Humidity (%)",
        "Rainfall": "Rainfall (mm)",
        "Max Temp": "Max Temperature (C)",
        "Min Temp": "Min Temperature (C)",
    },
    inplace=True,
)
df_filtered.head()

In [ ]:
# Reformat date

df_filtered["DATE"] = pd.to_datetime(df[["YEAR", "Month"]].assign(DAY=1))

# Filter out rows with Station Names = Dhaka

dhaka_weather = df_filtered.loc[df_filtered["Station Names"] == "Dhaka"]

# Sort the dataset in ascending order of dates

dhaka_weather = dhaka_weather.sort_values(by=["YEAR", "Month"], ascending=True)

# Drop the month and year columns as we already have a date column

dhaka_weather = dhaka_weather.drop(["Month", "YEAR"], axis=1)
dhaka_weather.head(12)

In [ ]:
# Convert dataset to CSV

dhaka_weather.to_csv(INPUT_PATHS["DHAKA_WEATHER_CSV"])

In [ ]:
# Importing and formatting historical Bitcoin data with Python

import csv
from datetime import datetime

timesteps = []
rainfall = []
with open(INPUT_PATHS["DHAKA_WEATHER_CSV"], "r") as f:
    csv_reader = csv.reader(f, delimiter=",")  # read in the target CSV
    next(csv_reader)  # skip first line (this gets rid of the column titles)
    for line in csv_reader:
        timesteps.append(
            datetime.strptime(line[7], "%Y-%m-%d")
        )  # get the dates as dates (not strings), strptime = string parse time
        rainfall.append(float(line[4]))  # get the closing price as float

# View first 12 of each

timesteps[:12], rainfall[:12]

In [ ]:
# Plot from CSV

import matplotlib.pyplot as plt

plt.figure(figsize=(30, 5))
plt.plot(timesteps, rainfall)
plt.title("Rainfall Pattern in Dhaka over the last 65 years", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Rainfall");

In [ ]:
# Get date and rainfall array
timesteps = dhaka_weather["DATE"].to_numpy()
rainfall = dhaka_weather["Rainfall (mm)"].to_numpy()

timesteps[:10], rainfall[:10]

In [ ]:
# Create train and test splits the right way for time series data

split_size = int(0.8 * len(rainfall))  # 80% train, 20% test

# Create train data splits (everything before the split)

X_train, y_train = timesteps[:split_size], rainfall[:split_size]

# Create test data splits (everything after the split)

X_test, y_test = timesteps[split_size:], rainfall[split_size:]

len(X_train), len(X_test), len(y_train), len(y_test)

In [ ]:
# Plot correctly made splits
plt.figure(figsize=(30, 5))
plt.plot(X_train, y_train, label="Train data")
plt.plot(X_test, y_test, label="Test data")
plt.xlabel("DATE")
plt.ylabel("Rainfall (mm)")
plt.legend(fontsize=14)
plt.show();

# Naive Forecast (Baseline)

In [ ]:
# Create a naïve forecast (Setting up a baseline for evaluating performance)

naive_forecast = y_test[:-1]  # Naïve forecast equals every value excluding the last value
naive_forecast[:10], naive_forecast[-10:]  # View frist 10 and last 10

In [ ]:
# Create a function to plot time series data

from experiment_weather_prediction import plot_time_series

In [ ]:
# Plot naive forecast

plt.figure(figsize=(30, 5))
plot_time_series(timesteps=X_train, values=y_train, label="Train data")
plot_time_series(timesteps=X_test, values=y_test, label="Test data")
plot_time_series(timesteps=X_test[1:], values=naive_forecast, format="-", label="Naive forecast");

In [ ]:
# Import tensorflow

import tensorflow as tf

In [ ]:
# MASE (Mean Absolute Scaled Error) implemented courtesy of sktime - https://github.com/alan-turing-institute/sktime/blob/ee7a06843a44f4aaec7582d847e36073a9ab0566/sktime/performance_metrics/forecasting/_functions.py#L16


In [ ]:
# Different Evaluation metrics in a single function

from experiment_weather_prediction import evaluate_preds

In [ ]:
# Evaluation of the Naive Forecast model

naive_results = evaluate_preds(y_true=y_test[1:], y_pred=naive_forecast)
naive_results

In [ ]:
# Setting window and horizon

HORIZON = 1  # predict 1 step at a time
WINDOW_SIZE = 11  # use  worth of timesteps to predict the horizon

In [ ]:
# Function to return array divided into window and horizon


In [ ]:
# Function to view NumPy arrays as windows

from experiment_weather_prediction import make_windows

In [ ]:
# testing the make windows function

full_windows, full_labels = make_windows(rainfall, window_size=WINDOW_SIZE, horizon=HORIZON)
len(full_windows), len(full_labels)

In [ ]:
# Make the train/test splits

from experiment_weather_prediction import make_train_test_splits

In [ ]:
# Splitting windows into training and test sets

train_windows, test_windows, train_labels, test_labels = make_train_test_splits(
    full_windows, full_labels
)
len(train_windows), len(test_windows), len(train_labels), len(test_labels)

In [ ]:
# Create a function to implement a ModelCheckpoint callback with a specific filename
from experiment_weather_prediction import create_model_checkpoint

# Basic LSTM Model (Univariate)

In [ ]:
# Before we pass our data to the LSTM layer, we have to reshape it in order to make sure it works


x = tf.constant(train_windows[0])
expand_dims_layer = tf.keras.layers.Lambda(
    lambda x: tf.expand_dims(x, axis=1)
)  # add an extra dimension for timesteps
print(f"Original shape: {x.shape}")  # (WINDOW_SIZE)
print(f"Expanded shape: {expand_dims_layer(x).shape}")  # (WINDOW_SIZE, input_dim)
print(f"Original values with expanded shape:\n {expand_dims_layer(x)}")

In [ ]:
tf.random.set_seed(42)

# Let's build an LSTM model with the Functional API
inputs = tf.keras.layers.Input(shape=(WINDOW_SIZE))
x = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, axis=1))(
    inputs
)  # expand input dimension to be compatible with LSTM
# print(x.shape)
# x = layers.LSTM(128, activation="relu", return_sequences=True)(x) # this layer will error if the inputs are not the right shape
x = tf.keras.layers.LSTM(256, activation="relu")(
    x
)  # using the tanh loss function results in a massive error
# print(x.shape)
# Add another optional dense layer (you could add more of these to see if they improve model performance)

output = tf.keras.layers.Dense(HORIZON)(x)
model_5 = tf.keras.Model(inputs=inputs, outputs=output, name="model_5_lstm")

# Compile model
model_5.compile(loss="mae", optimizer=tf.keras.optimizers.Adam())

# Seems when saving the model several warnings are appearing: https://github.com/tensorflow/tensorflow/issues/47554
model_5.fit(
    train_windows,
    train_labels,
    epochs=100,
    verbose=0,
    batch_size=128,
    validation_data=(test_windows, test_labels),
    callbacks=[create_model_checkpoint(model_name=model_5.name)],
)

In [ ]:
# Load in best version of model 5 and evaluate on the test data
model_5 = tf.keras.models.load_model("model_experiments/model_5_lstm.keras")
model_5.evaluate(test_windows, test_labels)

In [ ]:
from experiment_weather_prediction import make_preds

In [ ]:
# Make predictions with our LSTM model

model_5_preds = make_preds(model_5, test_windows)
model_5_preds[:10]

In [ ]:
# Evaluate model 5 preds

model_5_results = evaluate_preds(y_true=tf.squeeze(test_labels), y_pred=model_5_preds)
model_5_results

In [ ]:
offset = 0
plt.figure(figsize=(30, 5))
# Account for the test_window offset
plot_time_series(
    timesteps=X_test[-len(test_windows) :],
    values=test_labels[:, 0],
    start=offset,
    format="-",
    label="test_data",
)
plot_time_series(
    timesteps=X_test[-len(test_windows) :],
    values=model_5_preds,
    start=offset,
    format="-",
    label="model_5_preds",
)